# 📡 Google Fiber: BI Case Study - Dashboard de Análise

Este notebook automatiza a análise de **chamadas repetidas** para o Google Fiber, conforme planejado nas etapas de documentação de BI. 

**Desenvolvido por:** João Victor Póvoa França

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from datetime import datetime

# 1. Carregamento dos Dados
# Se estiver rodando no Colab, você pode subir o arquivo csv manualmente
try:
    df = pd.read_csv('google_fiber_data.csv')
    print("Dados carregados com sucesso!")
except:
    print("Arquivo não encontrado. Certifique-se de subir o 'google_fiber_data.csv' no Colab.")

# Converter data para datetime
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

## 🛠️ Processamento de Dados (Fase: Analisar)
Aqui calculamos as métricas de agregação temporal (Semana, Mês, Trimestre) solicitadas no Strategy Document.

In [ ]:
# Criar colunas de tempo
df['week'] = df['date'].dt.to_period('W').dt.to_timestamp()
df['month'] = df['date'].dt.to_period('M').dt.to_timestamp()
df['quarter'] = df['date'].dt.to_period('Q').dt.to_timestamp()

# Coluna consolidada de 'Total de Repetições' (soma de contatos n_1 a n_6)
repeat_cols = ['contacts_n_1', 'contacts_n_2', 'contacts_n_3', 'contacts_n_4', 'contacts_n_5', 'contacts_n_6']
df['total_repeats'] = df[repeat_cols].sum(axis=1)
df['has_repeat'] = (df['total_repeats'] > 0).astype(int)

## 📊 Visualizações (Fase: Monitorar)

### Gráfico 1: Repeat Calls by First Date
Este gráfico monitora a taxa de falha na resolução do primeiro contato ao longo do tempo.

In [ ]:
daily_res = df.groupby('month').agg({'contacts_n': 'sum', 'has_repeat': 'sum'}).reset_index()
daily_res['repeat_rate'] = (daily_res['has_repeat'] / daily_res['contacts_n'] * 100).round(2)

fig1 = px.line(daily_res, x='month', y='repeat_rate', 
               title='Tendência Mensal: Taxa de Chamadas Repetidas (%)', 
               labels={'repeat_rate': 'Taxa de Repetição (%)', 'month': 'Mês'},
               markers=True, template='plotly_dark')
fig1.show()

### Gráfico 2: Market and Problem Type (Foco Regionais)
Quais cidades e problemas geram mais reincidência imediata?

In [ ]:
market_pt = df.groupby(['market', 'problem_type'])['contacts_n_1'].sum().reset_index()

fig2 = px.bar(market_pt, x='market', y='contacts_n_1', color='problem_type', 
             barmode='group', title='Volume de Repetições no 1º Dia por Mercado e Tipo',
             labels={'contacts_n_1': 'Repetições (Dia 1)', 'market': 'Mercado'},
             template='plotly_dark')
fig2.update_layout(xaxis_title="Mercados", yaxis_title="Volume de Chamadas Repetidas")
fig2.show()

### Gráfico 3: Heatmap de Chamadas (Volume Total)
Visão geral 

In [ ]:
pivot_table = df.pivot_table(index='problem_type', columns='market', values='contacts_n', aggfunc='sum')

fig3 = px.imshow(pivot_table, text_auto=True, color_continuous_scale='Viridis',
                title='Concentração de Chamadas: Mercado vs Tipo de Problema',
                template='plotly_dark')
fig3.show()

### Gráfico 4: Agregações por Trimestre (Liderança)
Visão de alto nível para Emma e Keith (Stakeholders).

In [ ]:
q_res = df.groupby('quarter').agg({'contacts_n': 'sum', 'has_repeat': 'sum'}).reset_index()
q_res['quarter_str'] = q_res['quarter'].dt.strftime('Q%q %Y')

fig4 = px.bar(q_res, x='quarter_str', y=['contacts_n', 'has_repeat'], 
             title='Performance Trimestral: Volume Total vs Repetições',
             labels={'value': 'Volume', 'quarter_str': 'Trimestre'},
             barmode='group', template='plotly_dark')
fig4.show()